# Tutorial 8: Object Detection using Faster R-CNN (Custom Model)
Implement a custom Faster R-CNN model to detect inflamed joints in rheumatoid arthritis imagery.

In [ ]:
!pip install cython
!pip install opencv-python matplotlib
!pip install roboflow
!pip install 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 121.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13
  Cloning https://github.com/cocodataset/cocoapi.git to /tmp/pip-req-build-lz8gksj3
  Running command git clone --filter=blob:none --quiet https://github.com/cocodataset/cocoapi.git /tmp/pip-req-build-lz8gksj3
  Resolved https://github.co

## Data Fetching
Rheumatoid arthritis dataset labeled in Roboflow, replace the placeholders with your project details and API key.

In [ ]:
from roboflow import Roboflow
import os

# Initialize Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("workspace-name").project("rheumatoid-arthritis-joints")
# dataset = project.version(1).download("coco")

print("Please provide your Roboflow configuration to download the 150 labeled images.")

Please provide your Roboflow configuration to download the 150 labeled images.


## Faster R-CNN Model Definition
Torchvision to load a Faster R-CNN model with a ResNet50 backbone and modify the number of layers/classes for our custom task.

In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights

def get_model_custom(num_classes):
    # Load a model with pre-trained weights using the updated API
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)

    # Get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # Replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

# Example: 1 class (Inflamed Joint) + background = 2
num_classes = 2
model = get_model_custom(num_classes)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
print(f'Model loaded on {device}')

Model loaded on cpu


## Dataset and DataLoader
Define a class to load the images and annotations from the downloaded COCO JSON files.

In [ ]:
import os
import torch
from PIL import Image
from pycocotools.coco import COCO

class ArthritisDataset(torch.utils.data.Dataset):
    def __init__(self, root, annotation, transforms=None):
        self.root = root
        self.coco = COCO(annotation)
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.transforms = transforms

    def __getitem__(self, index):
        coco = self.coco
        img_id = self.ids[index]
        ann_ids = coco.getAnnIds(imgIds=img_id)
        coco_annotation = coco.loadAnns(ann_ids)
        path = coco.loadImgs(img_id)[0]['file_name']
        img = Image.open(os.path.join(self.root, path)).convert("RGB")

        num_objs = len(coco_annotation)
        boxes = []
        for i in range(num_objs):
            xmin = coco_annotation[i]['bbox'][0]
            ymin = coco_annotation[i]['bbox'][1]
            xmax = xmin + coco_annotation[i]['bbox'][2]
            ymax = ymin + coco_annotation[i]['bbox'][3]
            boxes.append([xmin, ymin, xmax, ymax])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((num_objs,), dtype=torch.int64)
        img_id = torch.tensor([img_id])

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = img_id

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.ids)

## Training Loop
This cell contains the basic training logic to optimize the Faster R-CNN model on dataset.

In [ ]:
from torchvision import transforms as T

def get_transform():
    return T.Compose([T.ToTensor()])

# Placeholder for training script
print("Once the dataset is downloaded, initialize ArthritisDataset and use torch.utils.data.DataLoader to start training.")

Once the dataset is downloaded, initialize ArthritisDataset and use torch.utils.data.DataLoader to start training.


## Inference and Visualization
Run the model on a single image and visualize the predicted bounding boxes.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_prediction(img_path, model, device, threshold=0.5):
    model.eval()
    img = Image.open(img_path).convert("RGB")
    transform = T.Compose([T.ToTensor()])
    img_tensor = transform(img).to(device).unsqueeze(0)

    with torch.no_grad():
        prediction = model(img_tensor)

    fig, ax = plt.subplots(1, figsize=(12, 9))
    ax.imshow(img)

    for box, score in zip(prediction[0]['boxes'], prediction[0]['scores']):
        if score > threshold:
            xmin, ymin, xmax, ymax = box.cpu().numpy()
            rect = patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin, linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
            ax.text(xmin, ymin, f'{score:.2f}', bbox=dict(facecolor='yellow', alpha=0.5))

    plt.show()

print("Ready for inference. Use `visualize_prediction(path, model, device)` once an image is available.")

Ready for inference. Use `visualize_prediction(path, model, device)` once an image is available.


## Evaluation Metrics
As per the tutorial objectives, we focus on:
1. **Intersection over Union (IoU)**: Measures the overlap between the predicted box and ground truth.
2. **Mean Average Precision (mAP)**: The standard metric for object detection, calculated across different IoU thresholds.